# 01 — Exploring Claude Code GitHub Issues

## Goal
Pull a sample of open issues and inspect the data shape before designing the categorization schema.

## 1. Setup

In [ ]:
import subprocess
import json
from pathlib import Path
from dotenv import load_dotenv
import os

load_dotenv()  # reads GITHUB_TOKEN from .env if present

REPO = "anthropics/claude-code"
DATA_DIR = Path("../data")
DATA_DIR.mkdir(exist_ok=True)

def gh(args: list[str]) -> list[dict]:
    result = subprocess.run(["gh"] + args, capture_output=True, text=True, check=True)
    return json.loads(result.stdout)

# Confirm auth
whoami = subprocess.run(["gh", "auth", "status"], capture_output=True, text=True)
print(whoami.stderr.strip())

## 2. Fetch a Sample

In [ ]:
issues = gh([
    "issue", "list",
    "--repo", REPO,
    "--state", "open",
    "--limit", "30",
    "--json", "number,title,body,labels,createdAt,updatedAt,comments,reactions,author,milestone"
])

print(f"Fetched {len(issues)} issues")

## 3. Inspect the Data

In [ ]:
# Top-level fields and types
sample = issues[0]
print("Fields:", list(sample.keys()))
print()
for k, v in sample.items():
    print(f"  {k}: {type(v).__name__} — {repr(v)[:80]}")

In [ ]:
# One full issue — readable view
print(f"#{sample['number']} — {sample['title']}")
print(f"Author : {sample['author']['login']}")
print(f"Labels : {[l['name'] for l in sample['labels']]}")
print(f"Created: {sample['createdAt']}")
print(f"Comments: {sample['comments']}")
print()
print("--- Body (first 500 chars) ---")
print((sample["body"] or "")[:500])

In [ ]:
# Label usage across the sample
from collections import Counter

all_labels = [l["name"] for issue in issues for l in issue["labels"]]
unlabeled = sum(1 for issue in issues if not issue["labels"])

print(f"Unlabeled issues: {unlabeled} / {len(issues)}")
print()
print("Label counts:")
for label, count in Counter(all_labels).most_common():
    print(f"  {count:>3}  {label}")

## 4. Save Raw Sample

In [ ]:
out = DATA_DIR / "raw_sample.json"
out.write_text(json.dumps(issues, indent=2))
print(f"Saved {len(issues)} issues → {out}")

## What I Learned

*(Fill in after running — note field structure, surprises, decisions)*

- **Fields available:** `number`, `title`, `body`, `labels`, `createdAt`, `updatedAt`, `comments`, `reactions`, `author`, `milestone`
- **Body quality:** 
- **Label coverage:** 
- **Surprises:** 
- **Decisions for categorization schema:** 